In [ ]:
#test_minmax
#See also: test_cloud_cover_local, test_vis_params, test_landsat_histogram

import ee
import geemap

# Initialize Earth Engine
ee.Initialize()

In [ ]:
# Load a Landsat 8 image (example: a single scene)
#image = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_044034_20210508') #SF
#image=ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140207') #GLBA Feb
#image=ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140919') #GLBA Sept (cloudy)
#image=ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_059019_20141021') #GLBA Oct (cloudy) and 28
image=ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140903') #Most clear Margerie, but scene has a lot of cloud

# Or use a filtered collection and take the first image
# collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA') \
#     .filterBounds(ee.Geometry.Point(-122.262, 37.871)) \
#     .filterDate('2021-01-01', '2021-12-31') \
#     .sort('CLOUD_COVER')
# image = collection.first()

# Select the bands (excluding thermal/panchromatic if desired)
bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11']
image = image.select(bands)

# Get image statistics: min and max per band
stats = image.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=image.geometry(),
    scale=30,
    maxPixels=1e9
)

# Print min and max for each band
#print("Band Min/Max Values:")
#for band in bands:
#    min_val = stats.get(f'{band}_min')
#    max_val = stats.get(f'{band}_max')
#    print(f"{band}: Min = {min_val}, Max = {max_val}")
#NOTE: that returns a complicated ee.ComputedObject. I want a simple number

#Pull the result to the client side → plain numbers
stats_dict = stats.getInfo()          # <-- forces server → client

print("Band   Min      Max")
print("-" * 25)
for b in bands:
    mn = stats_dict[f'{b}_min']
    mx = stats_dict[f'{b}_max']
    print(f"{b:<4} {mn:8.5f}  {mx:8.5f}")
    

In [ ]:
# Optional: Display on map
Map = geemap.Map()
Map.centerObject(image, 8)
Map.addLayer(image, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': .3 #0.3 for SF
}, 'Landsat 8 True 0-.3')
Map.addLayer(image, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 1.5 #0.3 for SF
}, 'Landsat 8 True 0-1.5')
Map.addLayer(image, {
    'bands': ['B4', 'B3', 'B2'],
    'min': -.5,
    'max': 1.5 #0.3 for SF
}, 'Landsat 8 True -.5-1.5 bring up shadows')
Map.addLayer(image, {
    'bands': ['B4', 'B3', 'B2'],
    'min': -.3,
    'max': 1.3 #0.3 for SF
}, 'Landsat 8 True -.3-1.3 bring up shadows blow out highlights')
#NOTE: Ideal for display would be something like a mosaic with range 0-.3 for darker areas and 0-1.5 for brighter 
#e.g. half histogram goes from 0-.15 and half .15-1.5. Not explaining this well... see image_visualization.ipynb
Map.addLayer(image, {
    'bands': ['B5', 'B4', 'B3'],
    'min': 0,
    'max': 1.0 #0.4 for SF
}, 'Landsat 8 False Color')
Map.addLayer(image, {
    'bands': ['B6', 'B5', 'B4'],
    'min': 0,
    'max': 1.0 #0.4 for SF
}, 'swir_nir_red_toa')
Map.addLayer(image, {
    'bands': ['B10'],
    'min': 230, #270-330 for SF
    'max': 280, #230-280 for GLBA winter
    "palette": ['blue', 'cyan', 'yellow', 'red']
}, 'thermal')
Map